# GUI 2: Extra Tools, Session Persistence, and Slash Commands

The first UI notebook shipped a working desktop (and web) application: a dark-themed chat window wired to the full agent stack. This notebook adds three independent layers on top of it.

**New tools.** Four builtin tools join the registry — `fetch_url`, `web_search`, `memory`, and `todo`. They follow the same `Tool` ABC as every other builtin, so they appear automatically in the agent's tool list and in approval dialogs with no changes to the agent loop or the UI.

**Session persistence.** `Session` gains `to_dict` / `from_dict` serialization and `save` / `load` / `list_saved` helpers that read and write JSON files under `~/.cda/sessions/`. A session stores the full message history, turn count, and cumulative token usage, so a conversation can be paused and resumed across process restarts.

**Slash commands.** The input field now doubles as a command line. Any message starting with `/` is intercepted in `_on_send` before it reaches the agent. A new module `ui/commands.py` handles the routing; the UI stays thin.

## New Builtin Tools

All four tools live in `src/notebooks/agent/tools/builtin/` and are registered in `ALL_BUILTIN_TOOLS`. They cover the two missing `ToolKind` categories — `NETWORK` and `MEMORY` — completing the original taxonomy.

### `fetch_url` and `web_search`

`FetchUrlTool` issues an async HTTP GET via `httpx`, strips HTML tags, and returns plain text capped at `max_length` characters (default 10,000). Light HTML stripping removes `<script>` and `<style>` blocks, replaces block-level tags with newlines, and collapses whitespace — no external HTML parser is required.

`WebSearchTool` queries the DuckDuckGo Instant Answer API, which requires no authentication. It returns up to `num_results` entries (default 5), each with a title snippet and URL. The API is rate-limit-friendly for reasonable interactive use.

Both tools have `kind = ToolKind.NETWORK`. Under `ApprovalPolicy.ON_REQUEST` they require user approval; under `AUTO` or `YOLO` they run unattended.

**Network tools.** Inspecting the new tool schemas:

In [ ]:
from notebooks.agent.tools.builtin.fetch_url import FetchUrlTool
from notebooks.agent.tools.builtin.web_search import WebSearchTool
from notebooks.agent.config import Config

config = Config()
for tool_cls in [FetchUrlTool, WebSearchTool]:
    t = tool_cls(config)
    schema = t.to_openai_schema()
    params = list(schema["parameters"]["properties"].keys())
    print(f"{t.name:15s} kind={t.kind.value:8s} params={params}")

**Fetch.** Retrieving the plain-text content of a URL:

In [ ]:
from pathlib import Path
from notebooks.agent.tools.base import ToolInvocation

fetch = FetchUrlTool(config)
result = await fetch.execute(ToolInvocation(
    params={"url": "https://example.com", "max_length": 500},
    cwd=Path.cwd(),
))
print(f"success={result.success}  chars={result.metadata.get('chars')}")
print(result.output[:300])

**Search.** Running a DuckDuckGo query:

In [ ]:
search = WebSearchTool(config)
result = await search.execute(ToolInvocation(
    params={"query": "Python asyncio tutorial", "num_results": 3},
    cwd=Path.cwd(),
))
print(f"success={result.success}  results={result.metadata.get('result_count')}")
print(result.output)

### `memory` and `todo`

`MemoryTool` is a persistent key-value store backed by `~/.cda/memory.json`. The agent calls it with one of five actions: `set`, `get`, `delete`, `list`, or `clear`. The file is written after every mutation. This lets the agent remember facts (API endpoints, user preferences, project paths) across separate sessions without cluttering the conversation context.

`TodoTool` manages a task list backed by `~/.cda/todos.json`. Each item has an auto-incrementing integer `id`, a `text` description, and a `done` flag. Actions: `add`, `done`, `delete`, `list`, `clear`. The agent uses this to track multi-step plans explicitly — listing the todo at the start of a session gives it a recoverable work queue.

Both tools have `kind = ToolKind.MEMORY`.

**Memory.** Store and retrieve a value, then list all entries:

In [ ]:
from notebooks.agent.tools.builtin.memory import MemoryTool
import tempfile, json
from pathlib import Path
from unittest.mock import patch

# Use a temp file so we don't pollute ~/.cda/memory.json during tests
tmp = Path(tempfile.mktemp(suffix=".json"))
mem = MemoryTool(config)

with patch.object(mem, "_store_path", return_value=tmp):
    for action, key, value in [
        ("set",  "project",   "ai-notebooks"),
        ("set",  "api_style",  "OpenRouter"),
        ("list", None,         None),
        ("get",  "project",    None),
    ]:
        params = {"action": action}
        if key:   params["key"]   = key
        if value: params["value"] = value
        r = await mem.execute(ToolInvocation(params=params, cwd=Path.cwd()))
        print(f"[{action}] {r.output}")

**Todo.** Create tasks, mark one done, then list:

In [ ]:
from notebooks.agent.tools.builtin.todo import TodoTool

tmp2 = Path(tempfile.mktemp(suffix=".json"))
todo = TodoTool(config)

with patch.object(todo, "_store_path", return_value=tmp2):
    steps = [
        {"action": "add",  "text": "Implement fetch_url tool"},
        {"action": "add",  "text": "Add session persistence"},
        {"action": "add",  "text": "Write slash command parser"},
        {"action": "done", "id": 1},
        {"action": "list"},
    ]
    for params in steps:
        r = await todo.execute(ToolInvocation(params=params, cwd=Path.cwd()))
        print(r.output)

### Updated registry

The default registry now contains eleven tools. We verify that all four new tools are present and their kinds are correct:

In [ ]:
from notebooks.agent.tools.registry import create_default_registry

registry = create_default_registry(config)
tools = registry.get_tools()
print(f"Total tools: {len(tools)}\n")
for t in tools:
    print(f"  {t.name:20s} {t.kind.value}")

## Session Persistence

`Session` is a runtime object — it holds the live message list, the `LLMClient`, and the `ToolRegistry`. Persistence serializes everything that can be restored from plain data: the message history, turn count, and token usage. The `Config` and the client / registry are not stored; they are always reconstructed from the environment on load.

The full API added to `Session`:

| Method | Description |
| :-- | :-- |
| `to_dict()` | Serialize to a JSON-compatible dict (version, saved_at, messages, usage, turn_count) |
| `from_dict(data, config)` | Class method — restore from a previously serialized dict |
| `save(name, directory?)` | Write `<directory>/<name>.json`; returns the `Path` written |
| `load(name, config, directory?)` | Class method — load from `<directory>/<name>.json` |
| `list_saved(directory?)` | Static method — sorted list of saved session names |

**Round-trip.** Create a session, add some messages, serialize, and restore:

In [ ]:
import tempfile
from pathlib import Path
from notebooks.agent.session import Session
from notebooks.agent.config import Config
from notebooks.agent.events import TokenUsage

config = Config()
session = Session(config)

# Simulate a short conversation
session.add_user_message("List files in the current directory")
session.add_assistant_message("I'll list the files for you.")
session.turn_count = 1
session.total_usage = TokenUsage(prompt_tokens=120, completion_tokens=30, total_tokens=150)

# Serialize
data = session.to_dict()
print(f"Serialized keys: {list(data.keys())}")
print(f"Messages stored: {len(data['messages'])} (system + user + assistant)")
print(f"Total tokens:    {data['total_usage']['total_tokens']}")

**Restore.** Calling `from_dict` rebuilds the session with the saved history:

In [ ]:
restored = Session.from_dict(data, config)

print(f"Turn count:     {restored.turn_count}")
print(f"Total tokens:   {restored.total_usage.total_tokens}")
print(f"Messages:       {len(restored.messages)}")
print(f"Last message:   {restored.messages[-1]['role']}: {restored.messages[-1]['content'][:60]}")

**Save and load.** Writing to disk and reading back:

In [ ]:
with tempfile.TemporaryDirectory() as tmpdir:
    save_dir = Path(tmpdir)

    # Save
    path = session.save("my_session", directory=save_dir)
    print(f"Saved to: {path.name}")

    # List
    names = Session.list_saved(directory=save_dir)
    print(f"Sessions: {names}")

    # Load
    loaded = Session.load("my_session", config, directory=save_dir)
    print(f"Loaded turn_count={loaded.turn_count}, messages={len(loaded.messages)}")

The default save directory is `~/.cda/sessions/` when no `directory` argument is given. Session names with spaces are stored with underscores (e.g. `"my project"` → `my_project.json`).

## Slash Commands

Rather than building a separate settings panel or toolbar, we repurpose the existing input field as a lightweight command line. Any message beginning with `/` is intercepted in `_on_send` *before* it reaches the agent. The routing lives in a new module `ui/commands.py` so `app.py` stays thin.

The full command set:

| Command | What it does |
| :-- | :-- |
| `/help` | List all commands |
| `/tools` | List registered tools with kind and description |
| `/stats` | Token usage, turn count, context window % |
| `/config` | Show model, approval policy, cwd, max turns |
| `/save [name]` | Save session to `~/.cda/sessions/<name>.json` |
| `/resume [name]` | Load session from disk and swap it into the agent |
| `/sessions` | List all saved sessions |
| `/clear` | Reset conversation history (keep config) |

An unknown slash command (e.g. `/foo`) falls through to the agent as a normal message, so the agent can handle it if it wants.

### The `handle_command` function

`handle_command(raw, agent, session)` parses the command, executes the operation, and returns a `CommandResult` dataclass. The UI reads the result and decides what to display or whether to swap the session:

In [ ]:
from notebooks.agent.ui.commands import CommandResult, handle_command
from notebooks.agent.agent import Agent
from notebooks.agent.session import Session
from notebooks.agent.config import Config

config = Config()
session = Session(config)
agent = Agent(config, session)

# Inspect the return type
import dataclasses
print("CommandResult fields:")
for f in dataclasses.fields(CommandResult):
    print(f"  {f.name}: {f.type}")

**`/help`.** Returns the full command listing as a system message:

In [ ]:
result = handle_command("/help", agent, session)
print(result.message)

**`/tools`.** Lists every registered tool:

In [ ]:
result = handle_command("/tools", agent, session)
print(result.message)

**`/stats`.** Reports token usage from the current session:

In [ ]:
from notebooks.agent.events import TokenUsage

session.turn_count = 3
session.total_usage = TokenUsage(prompt_tokens=4500, completion_tokens=800, total_tokens=5300)

result = handle_command("/stats", agent, session)
print(result.message)

**`/config`.** Shows the current configuration:

In [ ]:
result = handle_command("/config", agent, session)
print(result.message)

**`/save` and `/resume`.** These round-trip a session through disk. We use a temporary directory to avoid writing to `~/.cda/sessions/` during notebook execution:

In [ ]:
import tempfile
from pathlib import Path
from unittest.mock import patch
import notebooks.agent.session as sess_module

with tempfile.TemporaryDirectory() as tmpdir:
    save_dir = Path(tmpdir)

    # Patch the default sessions directory in both places it's used
    with patch.object(sess_module, "SESSIONS_DIR", save_dir):
        # /save
        save_result = handle_command("/save demo", agent, session)
        print("save:",   save_result.message)

        # /sessions
        list_result = handle_command("/sessions", agent, session)
        print("list:",   list_result.message)

        # /resume
        resume_result = handle_command("/resume demo", agent, session)
        print("resume:", resume_result.message)
        print("session_replaced:", resume_result.session_replaced)

When `session_replaced=True`, `app.py` swaps `agent.session` and refreshes the status bar counters:

In [ ]:
# Excerpt from _on_send (app.py) showing the session-swap logic:
snippet = '''
if text.startswith("/"):
    result = handle_command(text, self.agent, self.agent.session)
    if not result.not_a_command:
        if result.message:
            self._add_message(MessageData(role="system", content=result.message))
        if result.session_replaced and result.new_session is not None:
            self.agent.session = result.new_session
            self._turn = result.new_session.turn_count
            self._total_tokens = result.new_session.total_usage.total_tokens
            self._refresh_status()
        return
'''
print(snippet)

**Unknown command.** A command the parser doesn't recognize sets `not_a_command=True` and the caller forwards the text to the agent as a normal message:

In [ ]:
result = handle_command("/unknown_command", agent, session)
print(f"not_a_command={result.not_a_command}  message={result.message!r}")

## Updated Application Architecture

The UI package now contains three files. The architecture diagram from the previous notebook gains a command-intercept layer:

```
AgentApp (page controller)
├── Agent (agentic loop)
│   └── Session → LLMClient + ToolRegistry (11 tools)
├── ApprovalManager (approval decisions)
├── commands.py  ← NEW: slash command parser
└── Flet Page
    ├── StatusBar (model · turn · tokens)
    ├── ListView (message feed)
    │   ├── MessageBubble (user / assistant / system)
    │   └── ToolCallCard (tool invocations)
    └── InputBar (TextField + Send button)
          ↓ starts with "/"?
          ├── YES → handle_command() → system bubble / session swap
          └── NO  → agent.run()
```

The `CommandResult.session_replaced` flag is the only new coupling between `commands.py` and `app.py`. Everything else (new tools, session persistence) is additive — no existing code was modified beyond extending `Session` with new methods and adding tools to `ALL_BUILTIN_TOOLS`.

**Package public API.** The `ui` package now exports `CommandResult` and `handle_command` alongside the existing component factories:

In [ ]:
import notebooks.agent.ui as ui

print(ui.__all__)

## Running the App

The entry point is unchanged from the previous notebook. As a desktop app:

```{.bash filename="$ (local)"}
OPENROUTER_API_KEY=sk-... uv run flet run src/notebooks/agent/ui/app.py
```

As a web app served on `localhost:8550`:

```{.bash filename="$ (local)"}
OPENROUTER_API_KEY=sk-... uv run flet run --web --port 8550 src/notebooks/agent/ui/app.py
```

The input hint text now reads `Ask anything, or type /help for commands…` to surface the new feature. Type `/help` on first launch to see the full command list.

:::{.callout-note}
DuckDuckGo's Instant Answer API does not return results for all queries — complex or highly specific searches may return nothing. When that happens, `web_search` returns an error result and the agent can fall back to `fetch_url` with a direct URL or use the `shell` tool to run `curl`.

:::

---

■

← [GUI 1: Chat Interface](/notebooks/apps/cda/07-ui.html) &emsp; → [GUI 3: MCP Integration](/notebooks/apps/cda/09-ui3.html)